
# Step 01B — Semantic Attention Visualization Suite

This notebook creates the main visualization set for the A–E semantic-attention analysis.

It produces:

1. **100% semantic-attention composition plot**
2. **Raincloud-style distribution plots**
3. **Image × condition heatmaps**
4. **Connected per-advertisement trajectories**
5. **Effect-size / mean-difference forest plots**
6. **ILR compositional projection**

The notebook uses the balanced, QC-pass participant-level dataset whenever available.


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.linalg import helmert
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print("Working directory:", Path.cwd().resolve())


## 1. Locate and load the analysis dataset

In [ ]:

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for base in candidates:
        if (base / "semantic_attention_analysis_outputs").is_dir():
            return base

    for base in candidates[:4]:
        try:
            for child in base.iterdir():
                if child.is_dir() and (child / "semantic_attention_analysis_outputs").is_dir():
                    return child
        except Exception:
            pass

    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate semantic_attention_analysis_outputs. "
        "Set PROJECT_ROOT manually if needed."
    )

DATA_DIR = PROJECT_ROOT / "semantic_attention_analysis_outputs"

candidates = [
    DATA_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES_QC_PASS.csv",
    DATA_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES.csv",
    DATA_DIR / "master_semantic_attention_ALL_AVAILABLE_QC_PASS.csv",
]

DATA_FILE = next((p for p in candidates if p.exists()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "No master semantic-attention dataset found in "
        f"{DATA_DIR}"
    )

OUTPUT_DIR = Path.cwd().resolve() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)

required = ["participant_id", "condition", "image_id", "g_H", "g_P", "g_C", "g_O"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for c in ["g_H", "g_P", "g_C", "g_O"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "row_qc_pass" in df.columns:
    before = len(df)
    df = df[df["row_qc_pass"].astype(bool)].copy()
    print(f"QC-pass filter: {before} -> {len(df)} rows")

df["semantic_focus"] = 1.0 - df["g_O"]

CONDITIONS = ["A", "B", "C", "D", "E"]
df["condition"] = pd.Categorical(df["condition"], categories=CONDITIONS, ordered=True)

print("DATA_FILE:", DATA_FILE)
print("Rows:", len(df))
print("Images:", df["image_id"].nunique())
print("Participants:", df["participant_id"].nunique())
print("Conditions:", list(df["condition"].dropna().unique()))


## 2. Shared summaries used by the figures

In [ ]:

METRICS = ["g_H", "g_P", "g_C", "g_O", "semantic_focus"]
LABELS = {
    "g_H": "Headline",
    "g_P": "Product",
    "g_C": "CTA",
    "g_O": "Other / Background",
    "semantic_focus": "Semantic Focus (1 − g_O)",
}

condition_means = (
    df.groupby("condition", observed=True)[["g_H", "g_P", "g_C", "g_O", "semantic_focus"]]
      .mean()
      .reindex(CONDITIONS)
)

image_condition_means = (
    df.groupby(["image_id", "condition"], observed=True)[METRICS]
      .mean()
      .reset_index()
)

display(condition_means)



## 3. Figure 1 — 100% Semantic Attention Composition

Each condition is represented as a complete attention budget:

\[
g_H + g_P + g_C + g_O = 1.
\]

This makes it easy to see how the share of human fixation time is redistributed across Headline, Product, CTA, and Other/background.


In [ ]:

comp = condition_means[["g_H", "g_P", "g_C", "g_O"]].copy()

fig, ax = plt.subplots(figsize=(11, 6))

left = np.zeros(len(comp))
x = np.arange(len(comp))

for col in ["g_H", "g_P", "g_C", "g_O"]:
    vals = comp[col].to_numpy()
    ax.barh(x, vals, left=left, label=LABELS[col])
    left += vals

ax.set_yticks(x)
ax.set_yticklabels(comp.index)
ax.set_xlim(0, 1)
ax.set_xlabel("Proportion of total fixation time")
ax.set_ylabel("Condition")
ax.set_title("Semantic Attention Composition Across Conditions")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig01_semantic_attention_composition.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "fig01_semantic_attention_composition.pdf", bbox_inches="tight")
plt.show()



## 4. Figure 2 — Raincloud-style distributions

For each metric, the plot combines:

- a violin distribution;
- a boxplot;
- participant-level observations with slight horizontal jitter.

This helps distinguish a genuine distributional shift from a mean driven by a few observations.


In [ ]:

rng = np.random.default_rng(42)

for metric in METRICS:
    groups = [
        df.loc[df["condition"] == cond, metric].dropna().to_numpy()
        for cond in CONDITIONS
    ]

    fig, ax = plt.subplots(figsize=(10, 6))

    positions = np.arange(1, len(CONDITIONS) + 1)

    # Violin
    parts = ax.violinplot(
        groups,
        positions=positions,
        widths=0.8,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    # Box
    ax.boxplot(
        groups,
        positions=positions,
        widths=0.18,
        showfliers=False,
    )

    # Raw points
    for pos, values in zip(positions, groups):
        jitter = rng.normal(0, 0.045, size=len(values))
        ax.scatter(
            np.full(len(values), pos) + jitter,
            values,
            s=8,
            alpha=0.18,
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(CONDITIONS)
    ax.set_xlabel("Condition")
    ax.set_ylabel(LABELS[metric])
    ax.set_title(f"Distribution of {LABELS[metric]} Across Conditions")
    ax.grid(axis="y", alpha=0.2)

    fig.tight_layout()
    stem = f"fig02_raincloud_{metric}"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()



## 5. Figure 3A — Absolute image × condition heatmap

Each cell is the mean **Semantic Focus** for one advertisement under one condition.

\[
\text{Semantic Focus}=1-g_O.
\]


In [ ]:

heat_abs = (
    image_condition_means
    .pivot(index="image_id", columns="condition", values="semantic_focus")
    .reindex(columns=CONDITIONS)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(8, 10))

im = ax.imshow(heat_abs.to_numpy(), aspect="auto")

ax.set_xticks(np.arange(len(CONDITIONS)))
ax.set_xticklabels(CONDITIONS)
ax.set_yticks(np.arange(len(heat_abs.index)))
ax.set_yticklabels(heat_abs.index)

ax.set_xlabel("Condition")
ax.set_ylabel("Advertisement / Image ID")
ax.set_title("Semantic Focus by Advertisement and Condition")

for i in range(heat_abs.shape[0]):
    for j in range(heat_abs.shape[1]):
        value = heat_abs.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax, label="Mean Semantic Focus")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig03A_semantic_focus_heatmap_absolute.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "fig03A_semantic_focus_heatmap_absolute.pdf", bbox_inches="tight")
plt.show()



## 6. Figure 3B — Change from original Condition A

For each advertisement:

\[
\Delta SF_{i,c}=SF_{i,c}-SF_{i,A}.
\]

Positive values mean the condition directs a larger share of fixation time toward semantic AOIs than the original ad.


In [ ]:

delta = heat_abs.copy()

for cond in CONDITIONS:
    delta[cond] = heat_abs[cond] - heat_abs["A"]

delta_plot = delta[["B", "C", "D", "E"]]

fig, ax = plt.subplots(figsize=(8, 10))

max_abs = np.nanmax(np.abs(delta_plot.to_numpy()))
if not np.isfinite(max_abs) or max_abs == 0:
    max_abs = 1

im = ax.imshow(
    delta_plot.to_numpy(),
    aspect="auto",
    vmin=-max_abs,
    vmax=max_abs,
)

ax.set_xticks(np.arange(len(delta_plot.columns)))
ax.set_xticklabels(delta_plot.columns)
ax.set_yticks(np.arange(len(delta_plot.index)))
ax.set_yticklabels(delta_plot.index)

ax.set_xlabel("Condition compared with A")
ax.set_ylabel("Advertisement / Image ID")
ax.set_title("Change in Semantic Focus Relative to Original Condition A")

for i in range(delta_plot.shape[0]):
    for j in range(delta_plot.shape[1]):
        value = delta_plot.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:+.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax, label="Δ Semantic Focus vs A")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig03B_semantic_focus_delta_vs_A.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "fig03B_semantic_focus_delta_vs_A.pdf", bbox_inches="tight")
plt.show()



## 7. Figure 4 — Connected per-advertisement trajectories

Each thin line is one base advertisement.

The thicker black line is the overall image-level mean.

This figure is useful for answering:

> Is a condition effect consistent across advertisements, or is it driven by only a few stimuli?


In [ ]:

trajectory_metric = "semantic_focus"

wide = (
    image_condition_means
    .pivot(index="image_id", columns="condition", values=trajectory_metric)
    .reindex(columns=CONDITIONS)
    .dropna(how="any")
)

fig, ax = plt.subplots(figsize=(11, 7))

x = np.arange(len(CONDITIONS))

for image_id, row in wide.iterrows():
    ax.plot(x, row.to_numpy(), marker="o", linewidth=1, alpha=0.35)
    ax.text(x[-1] + 0.04, row.iloc[-1], str(image_id), fontsize=7, alpha=0.65)

overall = wide.mean(axis=0).to_numpy()
ax.plot(x, overall, marker="o", linewidth=3, label="Overall image-level mean")

ax.set_xticks(x)
ax.set_xticklabels(CONDITIONS)
ax.set_xlabel("Condition")
ax.set_ylabel("Mean Semantic Focus")
ax.set_title("Advertisement-level Semantic Focus Trajectories Across Conditions")
ax.grid(alpha=0.2)
ax.legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig04_semantic_focus_ad_trajectories.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "fig04_semantic_focus_ad_trajectories.pdf", bbox_inches="tight")
plt.show()



## 8. Figure 5 — Paired image-level effect forest plot

This figure focuses on **Condition C** compared with each other condition.

For each metric, the effect is:

\[
\Delta = \overline{Y_C - Y_X}
\]

computed after averaging participants within each advertisement × condition cell.

The 95% confidence interval is based on the paired advertisement-level differences.


In [ ]:

def paired_image_difference(metric, focal="C", comparator="A"):
    wide_metric = (
        image_condition_means
        .pivot(index="image_id", columns="condition", values=metric)
    )

    pair = wide_metric[[focal, comparator]].dropna()
    diff = pair[focal] - pair[comparator]

    n = len(diff)
    mean_diff = diff.mean()
    sd = diff.std(ddof=1)
    se = sd / np.sqrt(n)

    if n > 1:
        tcrit = stats.t.ppf(0.975, df=n-1)
        lo = mean_diff - tcrit * se
        hi = mean_diff + tcrit * se
        p = stats.ttest_rel(pair[focal], pair[comparator]).pvalue
    else:
        lo = np.nan
        hi = np.nan
        p = np.nan

    return {
        "metric": metric,
        "focal": focal,
        "comparator": comparator,
        "n_images": n,
        "mean_difference": mean_diff,
        "ci95_low": lo,
        "ci95_high": hi,
        "paired_t_p": p,
    }

forest_rows = []

for metric in METRICS:
    for comparator in ["A", "B", "D", "E"]:
        forest_rows.append(
            paired_image_difference(metric, focal="C", comparator=comparator)
        )

forest_df = pd.DataFrame(forest_rows)
forest_df.to_csv(OUTPUT_DIR / "fig05_effect_forest_data.csv", index=False)

for metric in METRICS:
    sub = forest_df[forest_df["metric"] == metric].copy()
    labels = [f"C − {c}" for c in sub["comparator"]]

    y = np.arange(len(sub))
    effects = sub["mean_difference"].to_numpy()
    lo = sub["ci95_low"].to_numpy()
    hi = sub["ci95_high"].to_numpy()

    xerr = np.vstack([effects - lo, hi - effects])

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(
        effects,
        y,
        xerr=xerr,
        fmt="o",
        capsize=4,
        linewidth=1.5,
    )
    ax.axvline(0, linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel(f"Mean paired image-level difference in {LABELS[metric]}")
    ax.set_title(f"Condition C Effect Across Advertisements — {LABELS[metric]}")
    ax.grid(axis="x", alpha=0.2)

    # p-values displayed as annotations, not used as the only interpretation.
    for yi, (_, row) in enumerate(sub.iterrows()):
        ax.text(
            hi[yi] if np.isfinite(hi[yi]) else effects[yi],
            yi,
            f"   p={row['paired_t_p']:.3g}",
            va="center",
            fontsize=8,
        )

    fig.tight_layout()
    stem = f"fig05_forest_C_vs_others_{metric}"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()

display(forest_df)



## 9. Figure 6 — ILR compositional projection

Because

\[
g_H+g_P+g_C+g_O=1,
\]

the four gaze shares form a **composition**.

We first replace exact zeros with a very small positive value, renormalize each composition, transform it to ILR coordinates, then project the 3-dimensional ILR space to 2 dimensions using PCA for visualization.

This figure is exploratory. Formal compositional inference belongs in Step 02.


In [ ]:

def zero_replace_and_close(X, eps=1e-6):
    X = np.asarray(X, dtype=float).copy()
    X[X <= 0] = eps
    X = X / X.sum(axis=1, keepdims=True)
    return X

def ilr_transform(X):
    """
    ILR using an orthonormal Helmert basis.
    Input: closed positive compositions, shape (n, D)
    Output: shape (n, D-1)
    """
    X = zero_replace_and_close(X)
    logX = np.log(X)
    basis = helmert(X.shape[1], full=False)  # (D-1, D)
    return logX @ basis.T

X = df[["g_H", "g_P", "g_C", "g_O"]].to_numpy()
Z = ilr_transform(X)

pca = PCA(n_components=2)
coords = pca.fit_transform(Z)

ilr_df = df[["condition", "image_id", "participant_id"]].copy()
ilr_df["PC1"] = coords[:, 0]
ilr_df["PC2"] = coords[:, 1]

fig, ax = plt.subplots(figsize=(10, 7))

for cond in CONDITIONS:
    sub = ilr_df[ilr_df["condition"] == cond]
    ax.scatter(
        sub["PC1"],
        sub["PC2"],
        s=12,
        alpha=0.16,
        label=cond,
    )

    # condition centroid
    cx = sub["PC1"].mean()
    cy = sub["PC2"].mean()
    ax.scatter(cx, cy, s=110, marker="X")
    ax.text(cx, cy, f"  {cond}", fontsize=10, va="center")

ax.set_xlabel(f"ILR-PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"ILR-PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Compositional Semantic Attention Space")
ax.grid(alpha=0.2)
ax.legend(title="Condition")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig06_ILR_compositional_projection.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "fig06_ILR_compositional_projection.pdf", bbox_inches="tight")
plt.show()

ilr_df.to_csv(OUTPUT_DIR / "fig06_ILR_projection_coordinates.csv", index=False)

print("Explained variance ratio:", pca.explained_variance_ratio_)



## 10. Extra — Small-multiple image trajectories for all five outcomes

This creates a separate trajectory figure for each semantic component, useful for identifying whether different AOIs react differently across advertisements.


In [ ]:

for metric in METRICS:
    wide_metric = (
        image_condition_means
        .pivot(index="image_id", columns="condition", values=metric)
        .reindex(columns=CONDITIONS)
        .dropna(how="any")
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(CONDITIONS))

    for _, row in wide_metric.iterrows():
        ax.plot(x, row.to_numpy(), marker="o", linewidth=0.9, alpha=0.28)

    overall = wide_metric.mean(axis=0).to_numpy()
    ax.plot(x, overall, marker="o", linewidth=3, label="Overall image-level mean")

    ax.set_xticks(x)
    ax.set_xticklabels(CONDITIONS)
    ax.set_xlabel("Condition")
    ax.set_ylabel(LABELS[metric])
    ax.set_title(f"Advertisement-level trajectories — {LABELS[metric]}")
    ax.grid(alpha=0.2)
    ax.legend()

    fig.tight_layout()
    stem = f"extra_trajectories_{metric}"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()


## 11. Output summary

In [ ]:

print("VISUALIZATION NOTEBOOK COMPLETE")
print("=" * 72)

files = sorted(OUTPUT_DIR.glob("*"))

print(f"Generated {len(files)} output files in:")
print(OUTPUT_DIR)
print()

for p in files:
    print(" -", p.name)

print("\nRecommended figures to inspect first:")
print(" 1. fig01_semantic_attention_composition.png")
print(" 2. fig03B_semantic_focus_delta_vs_A.png")
print(" 3. fig04_semantic_focus_ad_trajectories.png")
print(" 4. fig05_forest_C_vs_others_semantic_focus.png")
print(" 5. fig06_ILR_compositional_projection.png")
